# Debug Demo Tests

Notebook này dùng để smoke-test nhanh pipeline chatbot phim:

- Khởi tạo `NLUPipeline`, `TMDBRecommendationEngine`, `ConversationMemory`, `DialogManager`
- Test NLU trên nhiều câu mẫu
- Test hội thoại nhiều lượt để kiểm tra memory/context
- Test recommendation engine trực tiếp

Chạy lần lượt từ trên xuống dưới.

In [7]:
import sys, os

ROOT = r"d:\desktop\test"
os.chdir(ROOT)
# semantic_nlu.py dùng "from src.config" nên cần movie_chatbot_nlu trong path
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
MOD_ROOT = os.path.join(ROOT, "movie_chatbot_nlu")
if MOD_ROOT not in sys.path:
    sys.path.insert(0, MOD_ROOT)

# ── Setup ──────────────────────────────────────────────────────
from src.semantic_nlu import SemanticNLUInference

MODEL_PATH = os.path.join(ROOT, "movie_chatbot_nlu", "models", "semantic_model")
nlu = SemanticNLUInference(MODEL_PATH)
print("Model loaded.\n")


Loading SemanticNLU from d:\desktop\test\movie_chatbot_nlu\models\semantic_model...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3170.55it/s]
RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


OK SemanticNLU ready (cuda) | Slots: 6
Model loaded.



In [12]:
# ── Test cases: (câu hỏi, intent kỳ vọng) ─────────────────────
TEST_CASES = [
    # --- find_movie ---
    ("Tìm phim Ma của Christopher Nolan năm 2010",     "find_movie"),
    ("Cho tôi xem phim hành động có Bruce Willis",           "find_movie"),
    ("Phim tình cảm lãng mạn nào hay năm 2022?",             "find_movie"),

    # --- recommendation ---
    ("Gợi ý phim hay giống Inception",                       "recommendation"),
    ("Cho tôi xem gì đó vui vui",                            "recommendation"),
    ("Bạn hay xem phim gì vậy, recommend tôi với",           "recommendation"),

    # --- movie_info ---
    ("Cho tôi biết thêm về phim ký sinh trùng",                        "movie_info"),
    ("Phim Avengers ra năm mấy?",                            "movie_info"),
    ("Thời lượng của Interstellar là bao lâu?",              "movie_info"),

    # --- person_info ---
    ("Trấn Thành đóng những phim gì?",                       "person_info"),
    ("Diễn viên Leonardo DiCaprio nổi tiếng với phim nào?",  "person_info"),

    # --- genre_filter ---
    ("Có phim kinh dị nào mới ra gần đây không?",            "genre_filter"),
    ("Phim hành động bom tấn 2023",                          "genre_filter"),

    # --- greeting / goodbye ---
    ("Xin chào bạn ơi",                                      "greeting"),
    ("Tạm biệt nhé",                                         "goodbye"),

    # --- out of scope ---
    ("Hôm nay thời tiết thế nào?",                           "out_of_scope"),
    ("2 + 2 bằng mấy",                                       "out_of_scope"),
]

PASS = "✅"
FAIL = "❌"

print(f"{'#':<3} {'Intent OK':<10} {'Conf':<6} {'Slots':<30}  Câu hỏi")
print("─" * 90)

correct = 0
for i, (query, expected) in enumerate(TEST_CASES, 1):
    r = nlu.process(query)
    intent   = r["intent"]
    conf     = r["confidence"]
    args     = {k: v["value"] for k, v in r["arguments"].items()}
    ok       = PASS if intent == expected else FAIL
    if intent == expected:
        correct += 1
    slot_str = str(args) if args else "—"
    print(f"{i:<3} {ok} {intent:<20} {conf:<6.3f} {slot_str:<30}  {query}")

print("─" * 90)
print(f"\nKết quả: {correct}/{len(TEST_CASES)} đúng intent  ({correct/len(TEST_CASES)*100:.0f}%)")


#   Intent OK  Conf   Slots                           Câu hỏi
──────────────────────────────────────────────────────────────────────────────────────────
1   ✅ find_movie           0.698  {'genre': 'Ma', 'person': 'Christopher Nolan', 'year': '2010'}  Tìm phim Ma của Christopher Nolan năm 2010
2   ❌ genre_filter         0.678  {'genre': 'hành động'}          Cho tôi xem phim hành động có Bruce Willis
3   ❌ genre_filter         0.387  {'genre': 'tình cảm lãng mạn', 'year': '2022'}  Phim tình cảm lãng mạn nào hay năm 2022?
4   ✅ recommendation       0.679  {'like_movie': 'Inception'}     Gợi ý phim hay giống Inception
5   ❌ out_of_scope         0.964  —                               Cho tôi xem gì đó vui vui
6   ✅ recommendation       0.690  —                               Bạn hay xem phim gì vậy, recommend tôi với
7   ❌ person_info          0.688  {'name': 'trùng'}               Cho tôi biết thêm về phim ký sinh trùng
8   ✅ movie_info           0.614  {'title': 'Avengers'}           Phim

In [10]:
# ── Chi tiết output cho 3 câu đầu ─────────────────────────────
import json

DETAIL_QUERIES = [
    "Tìm phim kinh dị của Christopher Nolan năm 2010",
    "Gợi ý phim hay giống Inception",
    "Trấn Thành đóng những phim gì?",
]

for q in DETAIL_QUERIES:
    r = nlu.process(q)
    r_display = {k: v for k, v in r.items() if k != "query_vector"}
    print(f"Query   : {q}")
    print(json.dumps(r_display, ensure_ascii=False, indent=2))
    print()


Query   : Tìm phim kinh dị của Christopher Nolan năm 2010
{
  "input": "Tìm phim kinh dị của Christopher Nolan năm 2010",
  "intent": "find_movie",
  "confidence": 0.6721,
  "frame": "FIND_MOVIE",
  "arguments": {
    "genre": {
      "value": "kinh dị",
      "start": 9,
      "end": 16
    },
    "person": {
      "value": "Christopher Nolan",
      "start": 21,
      "end": 38
    },
    "year": {
      "value": "2010",
      "start": 43,
      "end": 47
    }
  },
  "entities": {
    "PERSON": [
      "Christopher Nolan"
    ],
    "GENRE": [
      "kinh dị"
    ],
    "YEAR": [
      "2010"
    ]
  },
  "hard_filters": {
    "PERSON": "Christopher Nolan",
    "YEAR": "2010",
    "GENRE": "kinh dị"
  },
  "ready_for_chapter2": true
}

Query   : Gợi ý phim hay giống Inception
{
  "input": "Gợi ý phim hay giống Inception",
  "intent": "recommendation",
  "confidence": 0.6795,
  "frame": "RECOMMENDATION",
  "arguments": {
    "like_movie": {
      "value": "Inception",
      "start": 